# 集成学习第二课：Bagging

## 本课目标

理解 Bagging 的核心思想：为什么“随机抽样 + 多模型平均 / 投票”能让模型更稳定。

一句话版：

```text
Bagging = Bootstrap 抽样 + 多个基学习器 + 投票 / 平均
```

## 1. 从单棵决策树的问题说起

单棵决策树的优点是直观、可解释，但它有一个明显问题：容易对训练数据过于敏感。

也就是说，训练数据稍微变一点，树的结构就可能变，最后预测结果也可能变。

这种现象可以理解为：

```text
单棵深树：方差高，不够稳定
```

Bagging 的想法是：

```text
不要只训练一个模型，而是训练很多个略有不同的模型，再把它们的结果合起来。
```

## 2. Bagging 这个词是什么意思？

Bagging 来自 Bootstrap Aggregating。

拆开看：

- Bootstrap：有放回抽样。
- Aggregating：聚合，也就是把多个模型结果合成一个最终结果。

所以 Bagging 的完整流程是：

```text
原始训练集
-> 有放回抽样，生成多份不同训练子集
-> 每份子集训练一个模型
-> 分类任务投票，回归任务平均
```

## 3. Bootstrap：有放回抽样

有放回抽样的意思是：每次抽到一个样本后，把它放回去，下一次还可能再抽到它。

假设原始训练集有 8 条样本：

```text
D = [1, 2, 3, 4, 5, 6, 7, 8]
```

某次 Bootstrap 抽样可能得到：

```text
D1 = [1, 1, 3, 4, 4, 6, 7, 8]
```

注意两件事：

- 有些样本会重复出现。
- 有些样本可能完全没被抽到。

In [1]:
# Bootstrap 有放回抽样演示
import numpy as np

rng = np.random.default_rng(22)
samples = np.arange(1, 9)

bootstrap_sample = rng.choice(samples, size=len(samples), replace=True)
not_selected = sorted(set(samples) - set(bootstrap_sample))

samples, bootstrap_sample, not_selected

(array([1, 2, 3, 4, 5, 6, 7, 8]),
 array([7, 3, 6, 2, 8, 1, 3, 6]),
 [np.int64(4), np.int64(5)])

上面输出里：

- 第一个数组是原始样本编号。
- 第二个数组是一次有放回抽样得到的新训练集。
- 第三个列表是这次没有被抽到的样本。

没有被某个模型抽到的样本，后面会和 OOB 袋外评估有关。

## 4. Bagging 为什么能提升稳定性？

Bagging 的关键不是简单地“模型多”，而是每个模型看到的数据不同。

这样训练出来的模型会有差异：

```text
模型 1 看到 D1
模型 2 看到 D2
模型 3 看到 D3
...
```

单个模型可能会因为某些训练样本而判断偏了，但多个模型一起投票或平均后，这种偶然性会被削弱。

从偏差-方差角度看：

```text
Bagging 主要降低方差，让模型预测更稳定。
```

## 5. 分类任务：多数投票

分类任务中，Bagging 通常让多个模型投票。

例如 5 个模型的预测结果是：

| 模型 | 预测 |
|---:|---|
| 模型 1 | A 类 |
| 模型 2 | B 类 |
| 模型 3 | A 类 |
| 模型 4 | A 类 |
| 模型 5 | B 类 |

最终结果是 A 类，因为 A 类票数更多。

In [ ]:
from collections import Counter

votes = ["A 类", "B 类", "A 类", "A 类", "B 类"]
vote_count = Counter(votes)
final_class = vote_count.most_common(1)[0][0]

vote_count, final_class

## 6. 回归任务：预测平均

回归任务中，Bagging 通常对多个模型的预测值取平均。

例如三个模型预测房价：

```text
98 万、102 万、100 万
```

最终预测：

```text
(98 + 102 + 100) / 3 = 100 万
```

In [ ]:
price_predictions = np.array([98, 102, 100])
price_predictions.mean()

## 7. sklearn 实战：单棵树 vs Bagging

下面用一个非线性分类数据集做对比。

我们比较：

- 单棵决策树
- Bagging + 多棵决策树

重点不是追求某一次分数最高，而是观察 Bagging 通常更稳定。

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=500, noise=0.3, random_state=22)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

tree = DecisionTreeClassifier(random_state=22)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

try:
    bagging = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        random_state=22,
        n_jobs=-1,
    )
except TypeError:
    bagging = BaggingClassifier(
        base_estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        random_state=22,
        n_jobs=-1,
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

tree_acc = accuracy_score(y_test, tree_pred)
bagging_acc = accuracy_score(y_test, bagging_pred)

tree_acc, bagging_acc

如果 Bagging 的准确率更高，不要只理解成“树多所以更强”。

更准确的理解是：

```text
多棵树看到的数据不同，错误不完全一样，投票后整体更稳。
```

## 8. 观察稳定性：多次随机划分

下面多次划分训练集和测试集，分别训练单棵树和 Bagging。

我们关注两件事：

- 平均准确率。
- 准确率标准差。

标准差越小，说明模型在不同数据划分下越稳定。

In [ ]:
tree_scores = []
bagging_scores = []

for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )

    tree = DecisionTreeClassifier(random_state=seed)
    tree.fit(X_train, y_train)
    tree_scores.append(accuracy_score(y_test, tree.predict(X_test)))

    try:
        bagging = BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=seed),
            n_estimators=100,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
        )
    except TypeError:
        bagging = BaggingClassifier(
            base_estimator=DecisionTreeClassifier(random_state=seed),
            n_estimators=100,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
        )

    bagging.fit(X_train, y_train)
    bagging_scores.append(accuracy_score(y_test, bagging.predict(X_test)))

summary = {
    "单棵树平均准确率": np.mean(tree_scores),
    "单棵树准确率标准差": np.std(tree_scores),
    "Bagging 平均准确率": np.mean(bagging_scores),
    "Bagging 准确率标准差": np.std(bagging_scores),
}

summary

## 9. OOB：袋外评估

Bootstrap 抽样时，每个基学习器都会有一些样本没抽到。

这些没被某个模型看到的样本，叫 Out-Of-Bag samples，简称 OOB。

直觉上：

```text
某个模型没见过这些样本，所以可以用它们来估计这个模型的泛化表现。
```

在 sklearn 中，可以设置：

```python
BaggingClassifier(oob_score=True)
```

训练后查看：

```python
model.oob_score_
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

try:
    bagging_oob = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        oob_score=True,
        random_state=22,
        n_jobs=-1,
    )
except TypeError:
    bagging_oob = BaggingClassifier(
        base_estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        oob_score=True,
        random_state=22,
        n_jobs=-1,
    )

bagging_oob.fit(X_train, y_train)

test_acc = accuracy_score(y_test, bagging_oob.predict(X_test))
oob_acc = bagging_oob.oob_score_

oob_acc, test_acc

OOB 分数不是测试集分数，但它能在不额外划分验证集的情况下，给出一个模型效果估计。

它特别适合 Bagging 这类使用 Bootstrap 抽样的方法。

## 10. Bagging 的重要参数

在 `BaggingClassifier` 中，常见参数有：

| 参数 | 含义 |
|---|---|
| `estimator` | 基学习器，例如决策树 |
| `n_estimators` | 基学习器数量 |
| `max_samples` | 每个基学习器抽多少样本 |
| `max_features` | 每个基学习器使用多少特征 |
| `bootstrap` | 是否对样本有放回抽样 |
| `bootstrap_features` | 是否对特征有放回抽样 |
| `oob_score` | 是否使用袋外样本评估 |
| `random_state` | 随机种子，保证结果可复现 |

最常调的是：

- `n_estimators`
- `max_samples`
- `max_features`
- 基学习器本身的参数，例如决策树的 `max_depth`

## 11. Bagging 和随机森林的关系

随机森林可以理解为 Bagging 在决策树上的增强版。

共同点：

- 都训练多棵树。
- 都使用 Bootstrap 样本抽样。
- 都通过投票或平均得到最终预测。

随机森林额外做了一件事：

```text
每个节点分裂时，只随机看一部分特征。
```

所以可以这样记：

```text
Bagging：样本随机。
随机森林：样本随机 + 特征随机。
```

## 12. 本课小结

- Bagging 来自 Bootstrap Aggregating。
- Bootstrap 是有放回抽样，会让每个基学习器看到不同训练数据。
- 分类任务通常投票，回归任务通常平均。
- Bagging 主要降低方差，让不稳定模型变得更稳定。
- OOB 可以利用没被抽到的样本估计模型效果。
- 随机森林可以看作 Bagging 的重要升级版。

记忆方式：

```text
Bagging = 抽很多份略不同的数据，训练很多个模型，最后一起表决。
```